In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader

In [2]:
# Example usage
embed_dim = 16  # Embedding dimension
num_heads = 4  # Number of attention heads
num_layers = 2  # Number of transformer layers
dropout = 0.1  # Dropout rate
graph_colomn=10
num_components=5
batch_size = 16  # Batch size

graph_input_dim = 4  # Number of colomns in the graph
text_vocab_size = 26  # Vocabulary size for text
graph_data = torch.randn(batch_size,graph_colomn, graph_input_dim)
text_input = torch.randint(0, text_vocab_size, (batch_size, num_components))
print("Graph data Shape",graph_data.shape)
print("First Graph data",graph_data[0])
print("Text Input Shape",text_input.shape)
print("First Text Input",text_input[0])

Graph data Shape torch.Size([16, 10, 4])
First Graph data tensor([[-0.7410, -0.4080, -1.6711,  0.5200],
        [-0.2563,  0.0444, -1.3415, -0.8594],
        [-0.5652,  0.6406, -2.3769,  0.6999],
        [-0.4042,  0.0984, -0.6089,  2.0119],
        [ 0.8920, -1.1323,  0.0297, -0.8781],
        [ 0.6018, -0.5184,  1.1979,  0.2992],
        [-0.1590,  1.3294,  1.2165,  2.7118],
        [ 1.3467, -1.9256,  0.0771, -0.6616],
        [ 0.1253, -0.4924,  0.2127, -0.4528],
        [-0.5156,  1.7504, -0.7627,  1.5003]])
Text Input Shape torch.Size([16, 5])
First Text Input tensor([12,  7, 22,  6,  1])


### Model

In [3]:
class GraphToTextTransformer(nn.Module):
    def __init__(self, graph_input_dim, text_vocab_size, embed_dim, num_heads, num_layers, dropout=0.1):
        super(GraphToTextTransformer, self).__init__()
        self.embed_dim = embed_dim

        # Encoder: Linear layer to embed graph columns
        self.encoder_embedding = nn.Linear(graph_input_dim, embed_dim)
    
        # Decoder: Embedding layer for text input
        self.decoder_embedding = nn.Embedding(text_vocab_size, embed_dim)


        # Transformer model
        self.transformer = nn.Transformer(
            d_model=embed_dim,
            nhead=num_heads,
            num_encoder_layers=num_layers,
            num_decoder_layers=num_layers,
            dropout=dropout
        )

        # Output layer to map decoder output to text vocabulary
        self.output_layer = nn.Linear(embed_dim, text_vocab_size)

    def forward(self, graph_data, text_input, src_mask=None, tgt_mask=None):
        """
        Args:
            graph_data: Tensor of shape (batch_size, seq_len, graph_input_dim)
            text_input: Tensor of shape (batch_size, tgt_seq_len)
            src_mask: Optional mask for the encoder input
            tgt_mask: Optional mask for the decoder input

        Returns:
            Tensor of shape (batch_size, tgt_seq_len, text_vocab_size)
        """
        # Encode graph data
        graph_encoded = self.encoder_embedding(graph_data)  # (batch_size, seq_len, embed_dim)
        graph_encoded = graph_encoded.permute(1, 0, 2)  # (seq_len, batch_size, embed_dim)

        # Embed text input
        text_embedded = self.decoder_embedding(text_input)  # (batch_size, tgt_seq_len, embed_dim)
        text_embedded = text_embedded.permute(1, 0, 2)  # (tgt_seq_len, batch_size, embed_dim)

        # Pass through transformer
        transformer_output = self.transformer(
            src=graph_encoded,
            tgt=text_embedded,
            src_mask=src_mask,
            tgt_mask=tgt_mask
        )  # (tgt_seq_len, batch_size, embed_dim)

        # Map to text vocabulary
        output = self.output_layer(transformer_output)  # (tgt_seq_len, batch_size, text_vocab_size)
        return output.permute(1, 0, 2)  # (batch_size, tgt_seq_len, text_vocab_size)

In [4]:
# Initialize model
model = GraphToTextTransformer(
    graph_input_dim, 
    text_vocab_size, 
    embed_dim, 
    num_heads, 
    num_layers, 
    dropout)

c:\Python313\Lib\site-packages\torch\nn\modules\transformer.py:385: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


In [5]:
learning_rate = 0.001
num_epochs = 10000

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

In [ ]:
# Training loop
for epoch in range(num_epochs):
    model.train()  # Set the model to training mode

    # Forward pass
    
    output = model(graph_data, text_input[:, :-1])  # Exclude the last token for input
    output = output.reshape(-1, text_vocab_size)  # Reshape for loss calculation
    # Take vocab index probabilistically


    target = text_input[:, 1:].reshape(-1)  # Exclude the first token for target

    # Compute loss
    loss = criterion(output, target)

    # Backward pass and optimization
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()


    # Print loss for the epoch
    if (epoch + 1) % 100 == 0:  # Print every 100 epochs
        print(f"Epoch [{epoch + 1}/{num_epochs}], Loss: {loss.item():.4f}")

Epoch [100/10000], Loss: 0.8218
Epoch [200/10000], Loss: 0.2627
Epoch [300/10000], Loss: 0.1110
Epoch [400/10000], Loss: 0.0596
Epoch [500/10000], Loss: 0.0460
Epoch [600/10000], Loss: 0.0297
Epoch [700/10000], Loss: 0.0236
Epoch [800/10000], Loss: 0.0237
Epoch [900/10000], Loss: 0.0124
Epoch [1000/10000], Loss: 0.0106
Epoch [1100/10000], Loss: 0.0080
Epoch [1200/10000], Loss: 0.1725
Epoch [1300/10000], Loss: 0.0080
Epoch [1400/10000], Loss: 0.0065
Epoch [1500/10000], Loss: 0.0180
Epoch [1600/10000], Loss: 0.0040
Epoch [1700/10000], Loss: 0.0035
Epoch [1800/10000], Loss: 0.0091
Epoch [1900/10000], Loss: 0.0029
Epoch [2000/10000], Loss: 0.0230
Epoch [2100/10000], Loss: 0.0031
Epoch [2200/10000], Loss: 0.0020
Epoch [2300/10000], Loss: 0.0015
Epoch [2400/10000], Loss: 0.0017
Epoch [2500/10000], Loss: 0.0012
Epoch [2600/10000], Loss: 0.0010
Epoch [2700/10000], Loss: 0.0013
Epoch [2800/10000], Loss: 0.0009
Epoch [2900/10000], Loss: 0.0012
Epoch [3000/10000], Loss: 0.0011
Epoch [3100/10000],

In [ ]:
# Predict the next letter
model.eval()  # Set the model to evaluation mode
with torch.no_grad():
    # Use the last token of the input sequence as the starting point
    input_token = torch.randint(0, text_vocab_size, (1, 1))  # Shape: (batch_size, 1)
    graph_data = torch.randn(1,1, graph_input_dim)
    print("Input token: ", input_token)
    # Pass through the model
    output = model(graph_data, input_token)
    print("Output ", output)
    # Get the predicted token (next letter) with the highest probability
    predicted_next_token = torch.argmax(output[:, -1, :], dim=-1)  # Shape: (batch_size,)
    print("Predicted next tokens:", predicted_next_token)

Input token:  tensor([[9]])
Output  tensor([[[ 2.6657, -1.8154,  0.1841,  2.7348,  0.6590,  5.4271, -0.2153,
          -1.5219, -2.5285, -0.2029,  4.8198, -1.6101,  1.0180, -1.4228,
           1.2086, -3.9604, -3.9229,  0.2807, -3.7280,  2.5446, -1.2577,
           0.0707,  1.3281, -4.0343, -0.4906,  1.1772]]])
Predicted next tokens: tensor([5])
